In [1]:
# ============================================================
# FILE STANDALONE - Densita' Turistica (comune x mese x anno)
# Stessa convenzione del notebook 01_pipeline_ingestion:
#   - RAW_DIR: file letti diretti da qui (niente sottocartella "originali")
#   - comune uniformato con .str.strip().str.title() (BLOCCO E)
#   - chiave di join: LOWER(TRIM(REPLACE(comune, apostrofo tipografico, apostrofo semplice)))
#   - filtro mese: le righe 'non disponibile' non vengono scartate, ma
#     ridistribuite sui 12 mesi secondo la stagionalita' reale degli
#     arrivi a porti/aeroporti (segnale regionale indipendente)
#   - soglia copertura mensile: stesso pattern delle celle Gini del notebook, qui = 8
# ============================================================
from pathlib import Path
import pandas as pd
import duckdb

# --- Percorso base del progetto (identico al notebook) ---
BASE_DIR = Path(r"C:\Users\giuse\sardegna-overtourism-aida26")
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "data" / "indicatore_densita"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DB_DIR = BASE_DIR / "db"
DB_PATH = DB_DIR / "sardegna_overtourism.duckdb"
DB_DIR.mkdir(parents=True, exist_ok=True)

# --- Connessione al database (stesso file del notebook) ---
con = duckdb.connect(str(DB_PATH))
con.execute("CREATE SCHEMA IF NOT EXISTS presentation")

ANNI = [2022, 2023, 2024, 2025]
SOGLIA_MESI_MINIMI = 10  # almeno 10 mesi su 12 con dati (stessa soglia usata per il Gini nel notebook)

# --- CSV utilizzati in questo blocco ---
FILE_SUPERFICIE = RAW_DIR / "superficie_comunale.csv"
FILE_PRESENZE = {a: RAW_DIR / f"csv_opendata_comuni_{a}.csv" for a in ANNI}
FILE_PORTI_AEROPORTI = RAW_DIR / "porti_aeroporti.csv"

print("CSV utilizzati:")
print(f"  {FILE_SUPERFICIE}")
for a in ANNI:
    print(f"  {FILE_PRESENZE[a]}")
print(f"  {FILE_PORTI_AEROPORTI}")


def chiave_comune(s):
    """Stessa chiave usata in staging.comuni_riferimento del notebook."""
    if pd.isna(s):
        return None
    apostrofo_tipografico = chr(0x2019)
    return str(s).strip().lower().replace(apostrofo_tipografico, "'")

CSV utilizzati:
  C:\Users\giuse\sardegna-overtourism-aida26\data\raw\superficie_comunale.csv
  C:\Users\giuse\sardegna-overtourism-aida26\data\raw\csv_opendata_comuni_2022.csv
  C:\Users\giuse\sardegna-overtourism-aida26\data\raw\csv_opendata_comuni_2023.csv
  C:\Users\giuse\sardegna-overtourism-aida26\data\raw\csv_opendata_comuni_2024.csv
  C:\Users\giuse\sardegna-overtourism-aida26\data\raw\csv_opendata_comuni_2025.csv
  C:\Users\giuse\sardegna-overtourism-aida26\data\raw\porti_aeroporti.csv


In [2]:
# ------------------------------------------------------------------
# 1) SUPERFICIE (anagrafica di riferimento per comune-superficie)
# ------------------------------------------------------------------
superficie = pd.read_csv(FILE_SUPERFICIE)
superficie["comune"] = superficie["comune"].astype(str).str.strip().str.title()
superficie["chiave_comune"] = superficie["comune"].map(chiave_comune)

print(f"\n=== FILE: {FILE_SUPERFICIE.name} ===")
print(f"righe: {len(superficie)} | comuni: {superficie['comune'].nunique()}")
print(superficie.head(3).to_string(index=False))


=== FILE: superficie_comunale.csv ===
righe: 377 | comuni: 377
   comune  superficie_kmq chiave_comune
   Modolo           2.616        modolo
  Tinnura           3.827       tinnura
Boroneddu           4.719     boroneddu


In [3]:

# ------------------------------------------------------------------
# 2) PESI DI STAGIONALITA' dagli arrivi porti/aeroporti (segnale regionale)
#    Servono per ridistribuire le presenze con mese "non disponibile"
# ------------------------------------------------------------------
porti_aeroporti = pd.read_csv(FILE_PORTI_AEROPORTI)
porti_aeroporti["data"] = pd.to_datetime(porti_aeroporti["data"])
porti_aeroporti["anno"] = porti_aeroporti["data"].dt.year
porti_aeroporti["mese"] = porti_aeroporti["data"].dt.month

arrivi_mensili = porti_aeroporti.groupby(["anno", "mese"], as_index=False)["arrivi"].sum()
arrivi_annuali = arrivi_mensili.groupby("anno")["arrivi"].transform("sum")
arrivi_mensili["peso_mese"] = (arrivi_mensili["arrivi"] / arrivi_annuali).round(6)

print(f"\n=== FILE: {FILE_PORTI_AEROPORTI.name} ===")
print("Pesi mensili di stagionalita' (esempio 2025):")
print(arrivi_mensili[arrivi_mensili["anno"] == 2025][["mese", "arrivi", "peso_mese"]].to_string(index=False))


=== FILE: porti_aeroporti.csv ===
Pesi mensili di stagionalita' (esempio 2025):
 mese  arrivi  peso_mese
    1  244915   0.030094
    2  228603   0.028089
    3  296888   0.036480
    4  569859   0.070021
    5  690289   0.084818
    6 1100374   0.135207
    7 1443956   0.177424
    8 1471737   0.180838
    9  925681   0.113742
   10  576728   0.070865
   11  259376   0.031871
   12  330028   0.040552


In [4]:

# ------------------------------------------------------------------
# 3) PRESENZE PER ANNO -> densita' turistica
# ------------------------------------------------------------------
tabelle_anno = {}

print("\n" + "=" * 68)
print("CALCOLO DENSITA' TURISTICA PER ANNO")
print("=" * 68)

for anno in ANNI:
    print(f"\n=== FILE: {FILE_PRESENZE[anno].name} ===")
    df = pd.read_csv(FILE_PRESENZE[anno], usecols=["comune", "mese", "presenze"])
    print(f"righe grezze: {len(df)}")

    # uniforma casing comune (BLOCCO E del notebook)
    df["comune"] = df["comune"].astype(str).str.strip().str.title()
    df["chiave_comune"] = df["comune"].map(chiave_comune)

    # righe con mese non assegnabile: NON vengono scartate, si sommano
    # per comune e verranno ridistribuite sui 12 mesi piu' avanti
    mask_non_disp = df["mese"].astype(str).str.strip().str.lower() == "non disponibile"
    non_allocabili = (
        df[mask_non_disp]
        .groupby("chiave_comune", as_index=False)["presenze"]
        .sum()
        .rename(columns={"presenze": "presenze_non_allocabili"})
    )
    perse = non_allocabili["presenze_non_allocabili"].sum()
    totale_anno = df["presenze"].sum()
    print(f"  presenze con mese non disponibile: {perse:,.0f} "
          f"({perse / totale_anno * 100:.3f}% del totale {anno}) -> verranno ridistribuite")

    # righe con mese valido
    df_mensile = df[~mask_non_disp].copy()
    df_mensile["mese"] = df_mensile["mese"].astype(int)

    agg = (
        df_mensile.groupby(["chiave_comune", "mese"], as_index=False)["presenze"]
        .sum()
        .rename(columns={"presenze": "presenze_mese"})
    )

    # ridistribuzione delle presenze non allocabili sui 12 mesi,
    # secondo i pesi di stagionalita' reale (arrivi porti/aeroporti)
    pesi_anno = arrivi_mensili[arrivi_mensili["anno"] == anno][["mese", "peso_mese"]]
    redistrib = non_allocabili.merge(pesi_anno, how="cross")
    redistrib["presenze_redistribuite"] = (
        redistrib["presenze_non_allocabili"] * redistrib["peso_mese"]
    ).round(2)
    redistrib = redistrib[["chiave_comune", "mese", "presenze_redistribuite"]]

    m = agg.merge(redistrib, on=["chiave_comune", "mese"], how="outer")
    m["presenze_mese"] = m["presenze_mese"].fillna(0)
    m["presenze_redistribuite"] = m["presenze_redistribuite"].fillna(0)

    # marca quali righe sono (in parte) stimate, prima di sommare
    m["presenze_stimate"] = m["presenze_redistribuite"] > 0
    m["presenze_mese"] = (m["presenze_mese"] + m["presenze_redistribuite"]).round(2)
    m = m.drop(columns=["presenze_redistribuite"])

    # join con superficie
    m = m.merge(
        superficie[["chiave_comune", "comune", "superficie_kmq"]],
        on="chiave_comune", how="left"
    )
    non_agganciati = m["comune"].isna().sum()
    if non_agganciati:
        print(f"  [!] {anno}: {non_agganciati} righe senza match superficie")

    m["anno"] = anno
    m["densita_turistica"] = (m["presenze_mese"] / m["superficie_kmq"]).round(2)

    # copertura mensile - stesso pattern delle celle Gini del notebook, soglia = 8
    m["n_mesi_con_dati"] = m.groupby("comune")["mese"].transform("nunique")
    m["copertura_sufficiente"] = m["n_mesi_con_dati"] >= SOGLIA_MESI_MINIMI

    out = (
        m[["comune", "anno", "mese", "presenze_mese", "presenze_stimate",
           "superficie_kmq", "densita_turistica",
           "n_mesi_con_dati", "copertura_sufficiente"]]
        .sort_values(["comune", "mese"])
        .reset_index(drop=True)
    )
    tabelle_anno[anno] = out

    percorso_out = OUT_DIR / f"densita_turistica_{anno}.csv"
    out.to_csv(percorso_out, index=False, encoding="utf-8-sig")

    print(f"{anno}: {out['comune'].nunique()} comuni | {len(out)} righe comune-mese "
          f"| salvato -> {percorso_out}")


CALCOLO DENSITA' TURISTICA PER ANNO

=== FILE: csv_opendata_comuni_2022.csv ===
righe grezze: 75448
  presenze con mese non disponibile: 110,667 (0.675% del totale 2022) -> verranno ridistribuite
  [!] 2022: 12 righe senza match superficie
2022: 189 comuni | 2180 righe comune-mese | salvato -> C:\Users\giuse\sardegna-overtourism-aida26\data\indicatore_densita\densita_turistica_2022.csv

=== FILE: csv_opendata_comuni_2023.csv ===
righe grezze: 98370
  presenze con mese non disponibile: 6,354 (0.039% del totale 2023) -> verranno ridistribuite
  [!] 2023: 12 righe senza match superficie
2023: 288 comuni | 3375 righe comune-mese | salvato -> C:\Users\giuse\sardegna-overtourism-aida26\data\indicatore_densita\densita_turistica_2023.csv

=== FILE: csv_opendata_comuni_2024.csv ===
righe grezze: 110498
  presenze con mese non disponibile: 6,321 (0.033% del totale 2024) -> verranno ridistribuite
  [!] 2024: 12 righe senza match superficie
2024: 307 comuni | 3661 righe comune-mese | salvato -> C

In [5]:

# ------------------------------------------------------------------
# SCRITTURA NEL DATABASE - presentation.densita_turistica
# Concatena i 4 anni e sostituisce la tabella (CREATE OR REPLACE)
# ------------------------------------------------------------------
densita_completa = pd.concat(tabelle_anno.values(), ignore_index=True)

con.register("densita_turistica_temp", densita_completa)
con.execute("""
    CREATE OR REPLACE TABLE presentation.densita_turistica AS
    SELECT * FROM densita_turistica_temp
""")

verifica_db = con.execute("""
    SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni,
           COUNT(DISTINCT anno) AS n_anni
    FROM presentation.densita_turistica
""").df()
print("\n" + "=" * 68)
print("SCRITTURA NEL DATABASE - presentation.densita_turistica")
print("=" * 68)
print(verifica_db.to_string(index=False))



SCRITTURA NEL DATABASE - presentation.densita_turistica
 n_righe  n_comuni  n_anni
   13032       330       4


In [6]:

# ------------------------------------------------------------------
# VERIFICA: due comuni di riferimento
#   - Cagliari: nessuna presenza non allocabile -> deve restare invariato
#   - Ovodda: 852 presenze TUTTE non allocabili nel 2025 -> ora ridistribuite
# ------------------------------------------------------------------
for comune_verifica in ["Cagliari", "Ovodda"]:
    print("\n" + "=" * 68)
    print(f"VERIFICA - {comune_verifica}")
    print("=" * 68)
    for anno in ANNI:
        r = tabelle_anno[anno]
        r = r[r["comune"] == comune_verifica]
        if r.empty:
            print(f"\n--- {anno}: nessun dato ---")
            continue
        print(f"\n--- {anno} ---")
        print(r.to_string(index=False))


VERIFICA - Cagliari

--- 2022 ---
  comune  anno  mese  presenze_mese  presenze_stimate  superficie_kmq  densita_turistica  n_mesi_con_dati  copertura_sufficiente
Cagliari  2022     1        22269.0             False          84.916             262.25             12.0                   True
Cagliari  2022     2        28694.0             False          84.916             337.91             12.0                   True
Cagliari  2022     3        35251.0             False          84.916             415.13             12.0                   True
Cagliari  2022     4        55550.0             False          84.916             654.18             12.0                   True
Cagliari  2022     5        75619.0             False          84.916             890.52             12.0                   True
Cagliari  2022     6        86649.0             False          84.916            1020.41             12.0                   True
Cagliari  2022     7       107343.0             False         

In [7]:
# verifica finale leggendo Cagliari direttamente dal DB (non da pandas)
print("\n" + "=" * 68)
print("VERIFICA DAL DB - Cagliari 2025 (letto da presentation.densita_turistica)")
print("=" * 68)
print(con.execute("""
    SELECT comune, anno, mese, presenze_mese, densita_turistica
    FROM presentation.densita_turistica
    WHERE comune = 'Cagliari' AND anno = 2025
    ORDER BY mese
""").df().to_string(index=False))


VERIFICA DAL DB - Cagliari 2025 (letto da presentation.densita_turistica)
  comune  anno  mese  presenze_mese  densita_turistica
Cagliari  2025     1        35523.0             418.33
Cagliari  2025     2        39978.0             470.79
Cagliari  2025     3        49804.0             586.51
Cagliari  2025     4        92205.0            1085.84
Cagliari  2025     5       111005.0            1307.23
Cagliari  2025     6       125171.0            1474.06
Cagliari  2025     7       154003.0            1813.59
Cagliari  2025     8       175651.0            2068.53
Cagliari  2025     9       140679.0            1656.68
Cagliari  2025    10       105291.0            1239.94
Cagliari  2025    11        46587.0             548.62
Cagliari  2025    12        47538.0             559.82


In [8]:
con.close()
print("\nConnessione al database chiusa.")


Connessione al database chiusa.
